# Frequency Filtering Experiments
2D FFT, Butterworth LPF/HPF, spectrum visualization, and inverse FFT.

In [ ]:
import sys
sys.path.insert(0, '..')

import logging
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from filtering_frequency.fft_transform import FFTTransform
from filtering_frequency.butterworth import ButterworthFilter
from filtering_frequency.frequency_filter import FrequencyFilter

logging.basicConfig(level=logging.INFO)
PROJECT_ROOT = Path('..').resolve()
OUTPUT_DIR = PROJECT_ROOT / 'output' / '03_frequency_filtering'
PREPROC_DIR = PROJECT_ROOT / 'output' / '01_preprocessing' / 'normalized'

In [ ]:
sample_paths = sorted(PREPROC_DIR.glob('*.png'))
assert len(sample_paths) > 0, 'Run preprocessing notebook first!'
sample = cv2.imread(str(sample_paths[0]), cv2.IMREAD_GRAYSCALE)
print(f'Sample: {sample.shape}')

In [ ]:
# FFT Spectrum visualization
fft = FFTTransform()
fft_shifted = fft.compute_fft(sample)
spectrum = fft.get_magnitude_spectrum(fft_shifted)
phase = fft.get_phase_spectrum(fft_shifted)
reconstructed = fft.compute_ifft(fft_shifted)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (title, img) in zip(axes, [
    ('Original', sample),
    ('FFT Magnitude Spectrum', spectrum),
    ('Phase Spectrum', phase),
    ('Reconstructed (IFFT)', reconstructed),
]):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.suptitle('2D FFT Analysis', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fft_analysis.png', dpi=150)
plt.show()

In [ ]:
# LPF comparison across cutoff frequencies
cutoffs = [10, 30, 50, 80]
order = 2

fig, axes = plt.subplots(1, len(cutoffs) + 1, figsize=(18, 4))
axes[0].imshow(sample, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

for ax, d0 in zip(axes[1:], cutoffs):
    lpf = ButterworthFilter.from_image(sample, cutoff=d0, order=order, filter_type='lpf')
    result = lpf.apply(sample)
    ax.imshow(result, cmap='gray')
    ax.set_title(f'LPF D0={d0}', fontsize=9)
    ax.axis('off')

plt.suptitle(f'Butterworth LPF — Order {order}', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'butterworth_lpf_comparison.png', dpi=150)
plt.show()

In [ ]:
# HPF comparison
fig, axes = plt.subplots(1, len(cutoffs) + 1, figsize=(18, 4))
axes[0].imshow(sample, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

for ax, d0 in zip(axes[1:], cutoffs):
    hpf = ButterworthFilter.from_image(sample, cutoff=d0, order=order, filter_type='hpf')
    result = hpf.apply(sample)
    ax.imshow(result, cmap='gray')
    ax.set_title(f'HPF D0={d0}', fontsize=9)
    ax.axis('off')

plt.suptitle(f'Butterworth HPF — Order {order}', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'butterworth_hpf_comparison.png', dpi=150)
plt.show()

In [ ]:
# Frequency response visualization (filter masks)
h, w = sample.shape
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col, d0 in enumerate(cutoffs):
    for row, ftype in enumerate(['lpf', 'hpf']):
        filt = ButterworthFilter(h, w, d0, order, ftype)
        mask = filt.get_mask()
        axes[row, col].imshow(mask, cmap='hot', vmin=0, vmax=1)
        axes[row, col].set_title(f'{ftype.upper()} D0={d0}', fontsize=8)
        axes[row, col].axis('off')
plt.suptitle('Butterworth Filter Masks (Frequency Response)', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'filter_masks.png', dpi=150)
plt.show()

In [ ]:
# Batch process all images
all_paths = sorted(PREPROC_DIR.glob('*.png'))
print(f'Processing {len(all_paths)} images...')

ff = FrequencyFilter(cutoff=30.0, order=2)
saved = ff.batch_process(all_paths, OUTPUT_DIR, cutoff=30.0, order=2)
for stage, paths in saved.items():
    print(f'  {stage}: {len(paths)} images')